# 10 · Panorama: votos e % por região, UF, capital/interior e porte

**Escopo**: 1º turno de Presidente 2026 (snapshot de 05/10/2026, 02:59). BR ainda `and="p"`, 41 seções não instaladas e 3 municípios da BA com geração antiga (`data/known_issues.csv`).

**Denominadores**: todo % é **sobre votos válidos** do recorte (ponderado: soma de votos ÷ soma de válidos). Não há média simples de percentuais municipais.

**Exterior (`zz`)** é um recorte à parte: fica fora de todos os agregados "Brasil" e aparece só na última seção.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
b = cp.base_2026()
bb = base.apenas_brasil(b)  # 5.571 municípios brasileiros
cand = cp.candidatos_2026_br()
nac = cp.percentuais_nacionais(cand, bb)
print("municípios BR:", len(bb), "| eleitorado BR:", int(bb.eleitorado.sum()))

## Região (ponderado: soma de votos ÷ soma de válidos da região)

In [ ]:
reg = base.votos_por_grupo(cand, bb, ["regiao"])
print(
    reg[reg.nr_candidato.isin([13, 22])][
        ["regiao", "nr_candidato", "votos", "validos_grupo", "pct_sobre_validos_grupo"]
    ]
    .round(2)
    .to_string(index=False)
)
print(base.participacao_agregada(bb, ["regiao"]).round(2).to_string(index=False))

In [ ]:
tab = cp.ponderar_regioes(g.agrupar_outros_grafico(reg, "regiao", "pct_sobre_validos_grupo", nac))
fig = g.barras_por_candidato(
    tab,
    "regiao",
    "pct_sobre_validos_grupo",
    "Votos válidos por região (% sobre válidos da região, ponderado)",
    "% dos válidos da região",
)
g.salvar(fig, "panorama_regioes_pt_pl")
plt.show()

## UF (ponderado)

In [ ]:
uf = base.votos_por_grupo(cand, bb, ["uf", "regiao"])
pv = uf[uf.nr_candidato.isin([13, 22])].pivot_table(
    index=["regiao", "uf"], columns="nr_candidato", values="pct_sobre_validos_grupo"
)
pv.columns = ["PT (13) %", "PL (22) %"]
print(pv.round(2).to_string())

## Capital × interior (ponderado)

In [ ]:
cap = base.votos_por_grupo(cand, bb, ["eh_capital"])
cap = cap[cap.nr_candidato.isin([13, 22])].assign(
    grupo=lambda d: np.where(d.eh_capital, "capitais (27)", "interior (5.544)")
)
print(
    cap[["grupo", "nr_candidato", "votos", "validos_grupo", "pct_sobre_validos_grupo"]]
    .round(2)
    .to_string(index=False)
)
fig = g.barras_por_candidato(
    cap,
    "grupo",
    "pct_sobre_validos_grupo",
    "Capitais × interior (% sobre válidos do recorte)",
    "% dos válidos",
)
g.salvar(fig, "panorama_capitais_interior")
plt.show()

## Porte do município (faixa de eleitorado de 2026; exterior não entra)

In [ ]:
fx = base.votos_por_grupo(
    cand, bb.dropna(subset=["faixa"]).assign(faixa=lambda d: d.faixa.astype(str)), ["faixa"]
)
fx = fx[fx.nr_candidato.isin([13, 22])]
n_fx = bb.groupby(bb.faixa.astype(str)).size().rename("n_municipios")
print(
    fx.pivot_table(index="faixa", columns="nr_candidato", values="pct_sobre_validos_grupo")
    .round(2)
    .join(n_fx)
    .to_string()
)
fig = g.barras_por_candidato(
    fx,
    "faixa",
    "pct_sobre_validos_grupo",
    "Votos válidos por porte do município (% sobre válidos da faixa)",
    "% dos válidos",
)
g.salvar(fig, "panorama_faixa_eleitorado")
plt.show()

## Exterior (recorte à parte, NÃO somado ao Brasil)

In [ ]:
ex = b[b.uf == "zz"].assign(recorte="exterior")
ext = base.votos_por_grupo(cand[cand.uf == "zz"], ex, ["recorte"])
print(
    ext[ext.nr_candidato.isin([13, 22])][
        ["nr_candidato", "votos", "validos_grupo", "pct_sobre_validos_grupo"]
    ]
    .round(2)
    .to_string(index=False)
)
print(base.participacao_agregada(ex, ["recorte"]).round(2).T.to_string())